# Random Forest Regression
Each small decision tree predicts a number. The forest returns the average. Fixed bootstrap samples keep the example simple and repeatable.

In [1]:
sizes = [950, 1200, 1450, 1700, 2000]
prices = [160, 205, 255, 320, 380]

def average(numbers):
    total = 0
    for number in numbers:
        total = total + number
    return total / len(numbers)

def squared_error(numbers):
    center = average(numbers)
    error = 0
    for number in numbers:
        difference = number - center
        error = error + difference ** 2
    return error

## Train one small regression tree

In [2]:
def train_regression_stump(sample_sizes, sample_prices):
    unique_sizes = sorted(set(sample_sizes))
    best_tree = None
    best_error = float('inf')

    for i in range(len(unique_sizes) - 1):
        threshold = (unique_sizes[i] + unique_sizes[i + 1]) / 2
        left_prices = []
        right_prices = []

        for j in range(len(sample_sizes)):
            if sample_sizes[j] < threshold:
                left_prices.append(sample_prices[j])
            else:
                right_prices.append(sample_prices[j])

        error = squared_error(left_prices) + squared_error(right_prices)
        if error < best_error:
            best_error = error
            best_tree = {
                'threshold': threshold,
                'left_answer': average(left_prices),
                'right_answer': average(right_prices)
            }
    return best_tree

def tree_prediction(tree, size):
    if size < tree['threshold']:
        return tree['left_answer']
    return tree['right_answer']

## Build three trees with fixed bootstrap samples

In [3]:
bootstrap_indices = [
    [0, 1, 1, 3, 4],
    [0, 0, 2, 3, 4],
    [1, 2, 2, 3, 4]
]

forest = []
for tree_number in range(len(bootstrap_indices)):
    sample_sizes = []
    sample_prices = []
    selected_indices = bootstrap_indices[tree_number]

    for index in selected_indices:
        sample_sizes.append(sizes[index])
        sample_prices.append(prices[index])

    tree = train_regression_stump(sample_sizes, sample_prices)
    forest.append(tree)
    print('Tree', tree_number + 1, tree)

Tree 1 {'threshold': 1450.0, 'left_answer': 190.0, 'right_answer': 350.0}
Tree 2 {'threshold': 1200.0, 'left_answer': 160.0, 'right_answer': 318.3333333333333}
Tree 3 {'threshold': 1575.0, 'left_answer': 238.33333333333334, 'right_answer': 350.0}


## Average the tree predictions

In [4]:
test_sizes = [1280, 1460]
for test_size in test_sizes:
    predictions = []
    for tree in forest:
        predictions.append(tree_prediction(tree, test_size))

    final_prediction = average(predictions)
    print('Size:', test_size)
    print('Tree predictions:', predictions)
    print('Random forest prediction:', round(final_prediction, 3), 'k$')

Size: 1280
Tree predictions: [190.0, 318.3333333333333, 238.33333333333334]
Random forest prediction: 248.889 k$
Size: 1460
Tree predictions: [350.0, 318.3333333333333, 238.33333333333334]
Random forest prediction: 302.222 k$
